In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, precision_score, recall_score
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
import re
from sklearn import metrics
from sklearn.model_selection import GridSearchCV
import math
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from pathlib import Path
from dsc80_utils import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import FunctionTransformer, StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import plotly.express as px
pd.options.plotting.backend = 'plotly'
import matplotlib.pyplot as plt
import math
from sklearn.preprocessing import Binarizer

# Leangue of Legends 

**Name(s)**: Ailinna Zhang

**Website Link**: [My website link](https://ailinnastar.github.io/LeagueOfLegends/predict.html)

##  Introduction

The datasets taken from all of the professional League of Legends games that have taken place in 2023, consisted with 128,064 rows and 93 columns.

This analysis will investigate about "**If the most banded champion is picked by a team will the team have higher chance to win that game**". 

Often some champions are the top choices in banding, which made me curious about if the champion is picked luckily will those champion increase the chance of winning, as it will carry the game compared to other champions.

In [ ]:

data = '2023_LoL_esports_match_data_from_OraclesElixir.csv'
loldata = pd.read_csv(data)  
print(loldata.head().to_markdown(index=False))
loldata

## Data Cleaning and Exploratory Data Analysis

### Data Cleaning

Through observing the data, when the `position` row is other than `team`, rows indicating picked champion are all `NaN`, which will affect the later steps and not necessary to keep rows that are not `team` in the `position` column. 

In [ ]:
first_rows = loldata[['position', 'pick1', 'pick2', 'pick3', 'pick4','pick5']].iloc[:10]
last_rows = loldata[['position', 'pick1', 'pick2', 'pick3', 'pick4','pick5']].iloc[-2:]
result = pd.concat([first_rows, last_rows], axis=0).reset_index(drop=True)
print(result.iloc[-7:-1].to_markdown(index=False))

Therefore, we first limit the data set to only contain `position` that is `team`.

In [ ]:
all_pick_team = loldata[(loldata['pick1'].notnull()) &(loldata['position'] == 'team')].copy()
 
print(all_pick_team.head().to_markdown(index=False))

![image.png](attachment:image.png)
![image-3.png](attachment:image-3.png)
![image-2.png](attachment:image-2.png)

In [ ]:
all_pick_team.loc[(loldata['ban4'].isnull())&(loldata['league']=='LFL'), (['league', 'teamname','split', 'side','result','patch','ban1', 'ban2', 'ban3', 'ban4', 'ban5'])]
print(all_pick_team.loc[(loldata['ban4'].isnull())&(loldata['league'] == 'LFL'), (['league', 'teamname','split', 'side','result','patch','ban1', 'ban2', 'ban3', 'ban4', 'ban5'])].to_markdown(index=False))


To find out the most banded champion from each patch version, we first have to combine band columns in to a column `'ban_list'` as a list, while we also accumulate and find out which chmapion is banned the most in the whole data set.


In [ ]:
count_ban_champion = {}
def banlist(row):
    banl = []
    for i in ['ban1', 'ban2', 'ban3', 'ban4', 'ban5']:
        if not pd.isnull(row[i]):
            banl.append(row[i])
            count_ban_champion[row[i]] = count_ban_champion.get(row[i], 0)+1
    return banl


all_pick_team['ban_list'] = all_pick_team.apply(banlist, axis=1)

In [ ]:

band_frequency=pd.DataFrame(count_ban_champion.values(), index=count_ban_champion.keys(), columns=['band frequency']).sort_values('band frequency')
fig = px.bar(band_frequency, x='band frequency',
             y=band_frequency.index, height=3000, title='Horizontal Bar Chart for banned champion frequency')
fig.update_yaxes(title_text="Banned champion")

fig.show()
fig.write_html(
    'graph/datac-horizontal.html')

From the above exploration and the  horizontal bar chart, `Maokai` was the most banned champion in the whole data set. Hoewever in LOL each patch have `Buff` and `Nerf` on some champions, therefore to invaestigate the appopriateness that `Maokai` is the one actually banned most without the influence of patch, we look in to the distribution of the patch. 

While first we check is there missingness in `patch`.

In [ ]:

print(all_pick_team.loc[all_pick_team['patch'].isna(), ('gameid','patch')].to_markdown(index=False))

Since there are missingness in the patch, we have to fill the `NaN` by the most common patched used in the same date or hour. 

In [ ]:
all_pick_team['y-m-d h'] = all_pick_team['date'].apply(lambda x: re.findall(
    r'\d{4}-\d{2}-\d{2} \d{2}', x)[0])
all_pick_team['y-m-d'] = all_pick_team['date'].apply(lambda x: re.findall(
    r'\d{4}-\d{2}-\d{2}', x)[0])

for i in all_pick_team['y-m-d h'].unique():
    same_ymd_patchs = all_pick_team.loc[(
        all_pick_team['y-m-d h'] == i) & (all_pick_team['patch'].notnull()), 'patch']
    most_common_patch = same_ymd_patchs.value_counts()
    if len(most_common_patch)==0:
        same_ymd_patchs = all_pick_team.loc[(
            all_pick_team['y-m-d'] == i[:-3]) & (all_pick_team['patch'].notnull()),'patch']
        most_common_patch = same_ymd_patchs.value_counts()
        
    all_pick_team.loc[(all_pick_team['y-m-d h'] == i) &
                      (all_pick_team['patch'].isnull()), 'patch'] = most_common_patch.idxmax()
    
print('No missingness in patch column? : ' +str(all_pick_team['patch'].isna().sum()==0))


After we fill all the missingness in thepatch column, we can look in to to the distribution of `patch`s.

In [ ]:
print('number of patches:'+ str( all_pick_team['patch'].value_counts().count()))
all_pick_team['patch'].value_counts(normalize=True).to_frame().iloc[:3]
print(all_pick_team['patch'].value_counts(
    normalize=True).to_frame().iloc[:3].to_markdown())

patch `13.01` accounts the most portion 18% from 20 patchs. 
Now let's look in to the most banned champion in each patch, to see whether `Maokai` being dominant of the mist banned champion fron each patch, or by the influence of patch.

In [ ]:
def inban(banl):
    count_ban_champion = {}
    for i in banl:
        for j in i:
            count_ban_champion[j] = count_ban_champion.get(j, 0)+1
    most_banned = max(count_ban_champion, key=count_ban_champion.get)
    print(str(banl.name) +' : \''+ str(most_banned)+ '\' banned the most')
    return most_banned

all_pick_team['most_banned_champion'] =all_pick_team.groupby('patch')['ban_list'].transform(inban)
print(all_pick_team.groupby('patch')[
      'most_banned_champion'].value_counts().to_frame().to_markdown())

In [ ]:
def Maokai_frequency(group):
    sum=0
    for bans in group:
        sum+='Maokai' in bans
    return sum


makai_fre= all_pick_team.groupby(
    'patch')['ban_list'].apply(Maokai_frequency)
makai_fre=pd.DataFrame(makai_fre)
fig = px.bar(makai_fre, x=makai_fre.index,
             y='ban_list', title='Bar Chart for Maokai banned frequency from each patch', text_auto='.2s',width=1000)

fig.update_traces(textfont_size=12, textangle=0,
                  textposition="outside", cliponaxis=False)
fig.update_xaxes(title_text="Patch",
                 tickvals=makai_fre.index, tickmode='array')
fig.update_yaxes(title_text="Makai Ban Frequency")
fig.show()

fig.write_html(
    'graph/datac-barc.html')

From the result above, `Maokai` is the most banned champion because `Maokai` was the champion banned most from the most dominant patch `13.01`. Therefore, we should not set the most banned champion as `Maokai`, instead we set the most banned champion according to patch, specified in the column `most_banned_champion`.

Now we can create a column which indicates is the most banned champion specified in the column `most_banned_champion`, is in those picked champions, and assigned to a new column `most_banned_champion_picked`, consisted with `True` or `False`.

In [ ]:
def most_banned_pick(row):
    for i in ['pick1', 'pick2', 'pick3', 'pick4', 'pick5']:
        if row[i]==row['most_banned_champion']:
            return True
    return False
    

all_pick_team['most_banned_champion_picked'] = all_pick_team.apply(lambda x:most_banned_pick(x),axis=1)

In [ ]:
print(all_pick_team[['teamname', 'ban_list', 'most_banned_champion', 'most_banned_champion_picked']].head().to_markdown(index=False))

Using the column `most_banned_champion_picked` we create side by side box plot and heat map to explore how the the average results in each patch is distributed, while also observe is there a significance difference in mean between MBC picked or not. 

In [ ]:
def average_result(a_group):
    return (a_group.sum()/a_group.count())


all_pick_team['average_result'] = all_pick_team.groupby(
    ['patch','most_banned_champion_picked'])['result'].transform(average_result)


df1 = all_pick_team[all_pick_team['most_banned_champion_picked']]
df2 = all_pick_team[~all_pick_team['most_banned_champion_picked']]


fig = make_subplots(rows=1, cols=2, shared_xaxes=False)
fig.add_trace(
    go.Box(y=df1['average_result'], name='MBC Picked'),
    row=1, col=1
)
fig.add_trace(
    go.Box(y=df2['average_result'], name='MBC Not Picked'),
    row=1, col=2
)
fig.update_yaxes(title_text="Average result", row=1, col=1)
fig.update_layout(
    title_text="Side by Side Box Plot for Average ressult on MBC picked",
    boxmode='group'  
)

fig.show()

fig.write_html(
    'graph/datac-sbsb.html')

colorscale = [
    [0, 'purple'],
    [0.5, 'pink'],
    [1, 'red']
]

fig = go.Figure(data=go.Heatmap(
    x=all_pick_team["patch"],
    y=all_pick_team['most_banned_champion_picked'],
    z=all_pick_team['average_result'],
    colorscale=colorscale
))
fig.update_yaxes(title_text="MBC picked")
fig.update_xaxes(title_text="patch version",
                 tickvals=makai_fre.index, tickmode='array')

fig.update_layout(
    title_text="Heat map on average result and patch by MBC picked"
)
fig.show()
fig.write_html(
    'graph/datac-hm.html')

From these two plots we can observe the mean and the spreadbess,the overall color in both groups are different and we can see MBC picked have better result overall, which gives me a insight of there is some influence made by picking MBC on the winning rate.

---

### Interesting Aggregates

Since we are interested in the most banned champion, we use pivot table to see which league is the hardest and which is the easiest to get to select the most banned champion.

In [ ]:

observe = (
    all_pick_team.pivot_table(
        index='most_banned_champion_picked', columns='league', aggfunc='size').fillna(0)
)

observe = observe / observe.sum()
observe
print(observe.to_markdown(index=False))

In [ ]:
print(
    f'Most difficult league to pick the most banned champion is {observe.T[True].idxmin()} with {observe.T[True].min()*100}%')
print(
    f'Easiest league to pick the most banned champion is {observe.T[True].idxmax()} with {observe.T[True].max()*100}%')


The most difficult league to pick the most banned champion is `CDF` with **2.0%**

The easiest league to pick the most banned champion is `LJLA` with **30.0%**

In [ ]:
def Maokai_frequency(group):
    return 'Maokai' in group


all_pick_team['league_maokai'] = all_pick_team['ban_list'].apply(Maokai_frequency)

observe = (
    all_pick_team.pivot_table(
        index=all_pick_team['league_maokai'], columns=all_pick_team['league'], aggfunc='size').fillna(0)
)

observe = observe / observe.sum()
observe
print(observe.to_markdown(index=False))

In [ ]:
print(
    f'The least banned Maokai league is {observe.T[True].idxmin()} with {observe.T[True].min()*100}%')
print(
    f'The most banned Maokai league is {observe.T[True].idxmax()} with {observe.T[True].max()*100}%')

The least banned Maokai league is DCup with 0.0%

The most banned Maokai league is WLDs with 29.56204379562044%

From this we can see that Maokai is not the most 'most' banned champion acorss the leaugue and the most banned champion are highlly depend on other column or other factors.

## Exploratory PCA

We have many numeric columns (objectives, gold, damage, vision, etc.). **PCA** (Principal Component Analysis) reduces these to a few components that capture most of the variation. This lets us:
- **Visualize** team-level matches in 2D (PC1 vs PC2) and see if win/loss or other groups separate.
- **Explore** which aspects of the game (e.g. early gold vs objectives) drive the first components.
- **Try different colorings** (e.g. by `result`, `most_banned_champion_picked`, `league`, or `patch`) to find interesting structure.

You can change `color_by` or the list of columns below to explore other aspects of the data.

In [ ]:
# Numeric columns for PCA (team-level stats). Add/remove to explore different aspects.
PCA_COLUMNS = [
    'gamelength', 'teamkills', 'teamdeaths',
    'firstblood', 'firstdragon', 'firstherald', 'firstbaron', 'firsttower',
    'dragons', 'opp_dragons', 'elders', 'opp_elders',
    'barons', 'opp_barons', 'towers', 'opp_towers',
    'inhibitors', 'opp_inhibitors',
    'damagetochampions', 'dpm', 'damageshare',
    'totalgold', 'earned gpm', 'goldat10', 'goldat15',
    'golddiffat10', 'golddiffat15',
    'visionscore', 'vspm', 'controlwardsbought', 'wardsplaced',
]
from sklearn.decomposition import PCA

pca_cols = [c for c in PCA_COLUMNS if c in all_pick_team.columns]
X_pca = all_pick_team[pca_cols].replace([np.inf, -np.inf], np.nan)
# Keep columns with enough data; fill remaining NaNs with median
X_pca = X_pca.loc[:, X_pca.notna().mean() > 0.5].fillna(X_pca.median())

scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X_pca)
pca = PCA(n_components=min(10, X_pca.shape[1]))
pca.fit(X_scaled)

print("Variance explained by first components:")
print(pd.Series(pca.explained_variance_ratio_).to_string())
print("\nCumulative:", np.cumsum(pca.explained_variance_ratio_))
n_95 = np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.95) + 1
print(f"Components needed for 95% variance: {n_95}")

In [ ]:
# Project to 2 components for visualization
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)
pc_df = pd.DataFrame(
    X_pca_2,
    columns=['PC1', 'PC2'],
    index=all_pick_team.index,
)
pc_df['result'] = all_pick_team['result'].astype(str)
pc_df['most_banned_champion_picked'] = all_pick_team['most_banned_champion_picked'].astype(str)
pc_df['league'] = all_pick_team['league'].astype(str)

# Plot 1: PC1 vs PC2 colored by result (win/loss)
fig_result = px.scatter(
    pc_df,
    x='PC1',
    y='PC2',
    color='result',
    title='PCA of team-level match stats (PC1 vs PC2), colored by result',
    labels={'PC1': f'PC1 ({pca_2.explained_variance_ratio_[0]:.1%} var)', 'PC2': f'PC2 ({pca_2.explained_variance_ratio_[1]:.1%} var)'},
)
fig_result.update_layout(margin=dict(l=60, r=60, t=50, b=60))
fig_result.show()

In [ ]:
# Plot 2: colored by most_banned_champion_picked
fig_mbc = px.scatter(
    pc_df,
    x='PC1',
    y='PC2',
    color='most_banned_champion_picked',
    title='PCA colored by MBC picked (True/False)',
    labels={'PC1': f'PC1 ({pca_2.explained_variance_ratio_[0]:.1%} var)', 'PC2': f'PC2 ({pca_2.explained_variance_ratio_[1]:.1%} var)'},
)
fig_mbc.update_layout(margin=dict(l=60, r=60, t=50, b=60))
fig_mbc.show()

# Save for website (path relative to project folder)
fig_result.write_html('diagram/pca_plot.html')

In [ ]:
# Component loadings: which variables drive PC1 and PC2?
loadings = pd.DataFrame(
    pca_2.components_.T,
    columns=['PC1', 'PC2'],
    index=X_pca.columns,
)
loadings['PC1_abs'] = loadings['PC1'].abs()
loadings['PC2_abs'] = loadings['PC2'].abs()
print("Top 10 variables for PC1 (by absolute loading):")
print(loadings.nlargest(10, 'PC1_abs')[['PC1', 'PC2']])
print("\nTop 10 variables for PC2:")
print(loadings.nlargest(10, 'PC2_abs')[['PC1', 'PC2']])

**Explore further:** Try coloring by `league` (e.g. `color='league'` in the scatter), or by `patch` to see meta shifts. You can also add/remove columns in `PCA_COLUMNS` (e.g. add `ckpm`, `turretplates`, `golddiffat15`) to see how the first two components change. Saving `fig_result.write_html('diagram/pca_plot.html')` lets you embed the plot on your website.

##  Assessment of Missingness

### NMAR Analysis


As we mensioned previously that there are some ban column were are missing, in my opinion this is classified as `not missing at random (NMAR)`.

In professional LOL league, they often do not ban all five of the champions, this sometimes implies that the team have more confidence in winning against the other, or there are no necessity on bannding other champions, or even to increases entertainment value for streamers.

There are many reasons which can not be pridicted through any column which demonstrates that the missing values in `ban1 to 5` are `NMAR`.

To make missingness in the ban columns be `MAR` from `NMAR`, collect listen to the conversation between players, planned ban champions before match and all the players' mind actitivities will be needed, and make a column that specify the the reason fo not banning if the team do not ban all 5. 


### Missingness Dependency ###

#### Check if all infroamtion of game after started 15 mins [**`'goldat15'`** ,**` 'xpat15'`** ,**`'csat15'`** ,**`'opp_goldat15'`** , **`'opp_xpat15'`** ,**`'opp_csat15'`** ,**`'golddiffat15'`** ,**`'xpdiffat15'`** ,**` 'csdiffat15'`**,**`'killsat15'`** ,**`'assistsat15'`**,**`'deathsat15'`**, **` 'opp_killsat15'`**, **`'opp_assistsat15'`**,**`'opp_deathsat15'`**] are **dependent** on **`'gamelength'`**.

**Null hypothesis**: The missingess of all infroamtion of game after started 15 mins(`'XXX15'`) does not depend on `'gamelength'`. 

**Alternative hypothesis**: The missingess of all infroamtion of game after started 15 mins(`'XXX15'`) depends on `'gamelength'`. 

**Test statistics**: Total variances difference.



In [ ]:

with_team = loldata[(loldata['position'] == 'team')]
col = with_team.columns
activity_15 = [i for i in col if re.search(r'.*15$', i)]
with_team['activity_15'] = with_team.apply(
    lambda x: not x[activity_15].isnull().all(), axis=1)

time_gap = np.arange(0, math.ceil(with_team['gamelength'].max()/60), 10)



def split_time(x):
    for i in time_gap:
        if math.ceil(x/60) <= i:
            return i

with_team['game_min'] = with_team['gamelength'].apply(split_time)

n_repetitions = 1000
shuffled = with_team.copy()

tvds = []
for _ in range(n_repetitions):

    shuffled['activity_15'] = np.random.permutation(with_team['activity_15'])

    pivoted = (
        shuffled
        .pivot_table(index='activity_15', columns='game_min', aggfunc='size')
    )

    pivoted = pivoted / pivoted.sum()

    tvd = pivoted.diff(axis=1).iloc[:, -1].abs().sum() / 2
    tvds.append(tvd)
    
observe = (
    with_team
    .pivot_table(index='game_min', columns='activity_15', aggfunc='size').fillna(0)
)

observe = observe / observe.sum()
observed_tvd = observe.diff(axis=1).iloc[:, -1].abs().sum() / 2

fig = px.histogram(pd.DataFrame(tvds), x=0, nbins=50, histnorm='probability',
                   title='Empirical Distribution of the TVD')
fig.add_vline(x=observed_tvd, line_color='red', line_width = 2)
fig.add_annotation(text=f'<span style="color:red">Observed TVD = {round(observed_tvd, 2)}</span>',
                   x=2.3 * observed_tvd, showarrow=False, y=0.16)
fig.update_xaxes(title_text="Total Variation Distance")
display(fig)
fig.write_html(
    'graph/depend-tvd.html')
fig2=observe.plot(
    kind='barh', title='Game length interval by Missingness of 15 mins data points', barmode='group')
fig2.update_xaxes(title_text="Proportion of the missingness")
fig2.update_yaxes(title_text="Game length")
fig2.write_html(
    'graph/depend-obs15.html')
fig2

In [ ]:
(np.array(tvds) >= observed_tvd).mean()

p-value is **0.517**.

Therefore, we accept the null hypothesis, which concludes the missingness in all of the 15mins columns are **not dependent** on `'gamelength'`.

----
#### Check if souls **`'infernals',	'mountains', 'clouds', 'ocean', 'chemtechs', 'hextechs'`** is **dependent** on **`'patch'`**.

In [ ]:
souls = ['infernals', 'mountains', 'clouds', 'oceans','chemtechs', 'hextechs']

Let `'souls'` be all the souls apears in the data.

**Null hypothesis**: The missingess of `'souls'` does not depend on `'patch'`. 

**Alternative hypothesis**: The missingess of `'souls'` depends on `'patch'`. 

**Test statistics**: Total variances difference.

In [ ]:
souls = ['infernals',	'mountains', 'clouds', 'oceans',
         'chemtechs', 'hextechs']
all_pick_team['missing_souls'] = all_pick_team.apply(
    lambda x: not x[souls].isna().all(), axis=1)
n_repetitions = 1000
shuffled = all_pick_team.copy()

tvds = []
for _ in range(n_repetitions):

    shuffled['missing_souls'] = np.random.permutation(all_pick_team['missing_souls'])

    pivoted = (
        shuffled
        .pivot_table(index='patch', columns='missing_souls', aggfunc='size')
    )

    pivoted = pivoted / pivoted.sum()

    tvd = pivoted.diff(axis=1).iloc[:, -1].abs().sum() / 2
    tvds.append(tvd)
    
observe = (
    all_pick_team
    .pivot_table(index='patch', columns='missing_souls', aggfunc='size').fillna(0)
)


observe = observe / observe.sum()
observed_tvd = observe.diff(axis=1).iloc[:, -1].abs().sum() / 2
fig = px.histogram(pd.DataFrame(tvds), x=0, histnorm='probability',
                   title='Empirical Distribution of the TVD')
fig.add_vline(x=observed_tvd, line_color='red', line_width = 2)
fig.add_annotation(text=f'<span style="color:red">Observed TVD = {round(observed_tvd, 2)}</span>',
                   x=.9 * observed_tvd, showarrow=False, y=0.05)
fig.update_xaxes(title_text="Total Variation Distance")
display(fig)

fig.write_html(
    'graph/depend-tvdsoul.html')
fig2=observe.plot(
    kind='barh', title='Patch by Missingness of souls', barmode='group')
fig2.update_xaxes(title_text="Proportion of the missingness")
fig2.update_yaxes(tickvals=all_pick_team['patch'], tickmode='array')

fig2.write_html(
    'graph/depend-obssoul.html')
fig2

In [ ]:
(np.array(tvds) >= observed_tvd).mean()

The p-value is **0.0**.

Therefore, **reject** the null hypothesis, conludes the missingness in `'url'` **dependens** on `'league'`.

## Step 4: Hypothesis Testing

**Null hypothesis**: If the highest band champion is picked the chances of win will still stays the same.

**Alternative hypothesis**: If the highest band champion is picked the chances of win will increase.

**Test statistics**: Difference in Means for wins and losses given the most banded champion is picked.

**Method:** One-sided permutation test with 10000 simulations with significance level 0.05.



In [ ]:
n_repetitions = 10000

differences = []
for _ in range(n_repetitions):
    with_shuffled = all_pick_team.assign(
        Shuffled_picked_mbc=np.random.permutation(all_pick_team['most_banned_champion_picked']))
    group_means = (
        with_shuffled
        .groupby('Shuffled_picked_mbc')
        .mean()
        .loc[:, 'result']
    )
    difference = group_means.diff().iloc[-1]
    differences.append(difference)
differences

obsobserved_difference = all_pick_team.groupby('most_banned_champion_picked')[
    'result'].mean().diff().iloc[-1]
print(obsobserved_difference)

In [ ]:
fig = px.histogram(
    pd.DataFrame(differences), x=0, nbins=60, histnorm='probability',
    title='Empirical Distribution of the Mean Differences <br> in the most banned champion picked (Picked - Non-Picked)')
fig.add_vline(x=obsobserved_difference, line_color='red', line_width = 2)
fig.add_annotation(text=f'<span style="color:red">Observed = {round(obsobserved_difference, 2)}</span>',
                   x=.9 * obsobserved_difference, showarrow=False, y=0.07)
fig.update_xaxes(title_text="Differences in mean")

fig.show()
fig.write_html(
    'graph/hypothe-ditribution.html')

In [ ]:
(np.array(differences) >= obsobserved_difference).mean()

p-value: 0.0


We **reject the null hypothesis** that the highest band champion is picked the chances of win will still stays the same.
Therefore, picking the highest banned champion does influence the win rate.


## Step 5: Framing a Prediction Problem

**Prediction Problem:** Predict the match result for all the team from all professional League of Legends (LoL)

**Model Type:** It will be **Classification**, as the result will be eaither 1 for win, 0 for loss.

**Response variable:** The model will predict the `result`, as result is the most important element in the game as a streamer, so that we can understand what factors can contribute to higher win rate and improve ourselve in playing LOL. While similarly it is significant for the player to analyse team performance and strategy, which will lead to better decision and more successes.

**Evaluation Metric:** It will be evaluate on **accuracy**, which is the proportion of correct prediction out of all prediction. As the number of win and lose are well balanced, and the distribution is not skewed, metrics such as the F1-score, Precision, or Recall will not be used and not be appropriate.



## Step 6: Baseline Model

In [ ]:
from sklearn.metrics import mean_squared_error
from math import sqrt

def rmse(y_act, y_pre):
    return sqrt(mean_squared_error(y_act, y_pre))

**Model Description:**
**RandomForestClassifier** is the classifier chosen for this prodiction model prediting `result`(1 - win or 0 -lose) with **5 Norminal features.**

| Column name(s) | Data type |Encoding|Reason of chose|
|:-----------:|:-----------:|:-----------:|:-----------:|
| 'firstblood', 'firstbaron', 'firstdragon', 'firsttower' | Norminal|FunctionTransformer(add) - add all together| Early achivements are really curcial as it often demonstrates the differences in performance and harmony in the team, provides momentum and psychological advantages, which will directly affect the result.|
| 'most_banned_champion_picked'| Norminal|OneHotEncoder() - One Hot Encoded| From the hypothesis tessting, we concluded whether or not picking MBC will influence the win rate.| 







In [ ]:
features = [
    'firstblood', 'firstdragon', 'firstbaron', 'firsttower', 'most_banned_champion_picked'
]

def add(data):
    return pd.DataFrame(np.sum(data, axis=1))


X = all_pick_team
y = all_pick_team['result']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)
col_trans = ColumnTransformer([('add', FunctionTransformer(add, validate=False), ['firstblood', 'firstdragon', 'firstbaron', 'firsttower']), 
                               ('most_picked', OneHotEncoder(drop='first'), [
                                'most_banned_champion_picked']),

    ], remainder='passthrough')

plbase = Pipeline([
    ('transf', col_trans),
    ('RC', RandomForestClassifier(random_state=42))
])

plbase.fit(X_train[features], y_train)
print(f'train score: {plbase.score(X_train, y_train)}')
print(f'test score: {plbase.score(X_test, y_test)}')
print(f'train rmse: {rmse(plbase.predict(X_train), y_train)}')
print(f'test rmse: {rmse(plbase.predict(X_test), y_test)}')

Through only using these features, I think the model performs **well** with accuracy **73.97% in  train**, **72.65% in test** concidering with the fact that games results often comes with luck.

While **bad** in the same time, as we are predicting it is 1, 0 for result and the root mean squre error is .513 whcih means my predictions and the actual values is a bit over halfway between a perfect model and the worst-case scenario, which says it is not a ideal model.

In [ ]:
added_feature_name = ['sum_firsts']
ohe_feature_names = col_trans.named_transformers_[
    'most_picked'].get_feature_names_out(['most_banned_champion_picked'])
transformed_feature_names = added_feature_name + list(ohe_feature_names)
feature_importances = plbase.named_steps['RC'].feature_importances_
feature_importance_dict = {name: importance for name, importance in zip(
    transformed_feature_names, feature_importances)}
for name, importance in feature_importance_dict.items():
    print(f"{name}: {importance}")

If we look in to the measure of the importance or contribution of each feature used in the prediction, `sum_firsts` contributes dominantly with 99.6%, which is the column added up by `firstXX`, while suprisingly the one hot encoded`most_banned_champion_picked` did not contribute a lot to the prediction as I expacted.

## Step 7: Final Model

**Model Description:**
**RandomForestClassifier** is the classifier chosen for this prodiction model prediting `result`(1 - win or 0 -lose) with **16 features.**

| Column name(s) | Data type |Encoding|Reason of chose|
|:-----------:|:-----------:|:-----------:|:-----------:|
| 'firstblood', 'firstbaron', 'firstdragon', 'firsttower' | Norminal|FunctionTransformer(add) - add all together| Early achivements are really curcial as it often demonstrates the differences in performance and harmony in the team, provides momentum and psychological advantages, which will directly affect the result.|
| 'xpat15', 'goldat15'| Quantative| QuantileTransformer()| Since 0-30 mins are growingtime, 15 min is a curcial time stamp where teams have half finished building up the base, demonstrates how well the team is developoing agisnt the other, which will influence the result as well. As vales have wide range we used quualtile to split in to quarter. | 
| 'team kpm'| Quantative| Binarizer(threshold=all_pick_team['team kpm'].mean()),PolynomialFeatures(degree=2, include_bias=False)| This indicates how teams are efficient in kill the oponents, which evalueates to better players' technique and team stretegy which highly leads to sucess .| 
| 'earned gpm'| Quantative| PolynomialFeatures(degree=2, include_bias=False)| Depict how teams are efficiently growing economicly, so tha they can grow and purchase item faster to be more advantaged.| 
| 'dpm', 'wpm', 'wcpm', 'vspm','cspm'| Quantative|  PolynomialFeatures(degree=2, include_bias=False) |These evaluate the vision control and team strategy and decisions, how effecting utilising the charaterisctics of champion, and its position, which will be consist as a huge part to have success.   | 
| 'dragons', 'barons', 'towers'| Quantative| Pass in as it is |These measures reflect that teams stretigic decisgion, team strength, and team resources, which ofcourse contribute significantly to the result.|








In [ ]:


features = ['team kpm', 'dpm', 'wpm', 'wcpm', 'vspm', 'earned gpm', 'dragons', 'cspm',
            'barons', 'towers',  'xpat15', 'goldat15', 'firstblood', 'firstdragon', 'firstbaron', 'firsttower']

def add(data):
    return pd.DataFrame(np.sum(data, axis=1))


col_trans = ColumnTransformer([('add_first', FunctionTransformer(add, validate=False), ['firstblood', 'firstdragon', 'firstbaron', 'firsttower']),
                               ('quatile15', QuantileTransformer(),
                                ['xpat15', 'goldat15']),
                               ('org', FunctionTransformer(lambda x: x),
                                ['team kpm', 'earned gpm']),
    ('size', Binarizer(threshold=all_pick_team['team kpm'].mean()), ['team kpm']),
    ('poly', PolynomialFeatures(degree=2, include_bias=False), [
     'team kpm', 'earned gpm', 'dpm', 'wpm', 'wcpm', 'vspm', 'cspm'])
], remainder='passthrough')
pl = Pipeline([
    ('transf', col_trans),
    ('RC', RandomForestClassifier(random_state=6))
])

Run `GridSearchCV` with 5 folds on the model to find the hyper-parameter optimises the score for unseen datas.

Hyperparameters chosed to be tuned:

| Hyperparameters name |Reason of chose|
|:-----------:|:-----------:|
| `'n_estimators'` |Number of trees influence the model's stability and accuracy since the prediction is based on the majority from all tree result, but more trees means more time and cost therefore have to find the balance in the number of trees.|
| `'max_depth'` |As deeper trees can capture more complex patterns, however over complicated will leads to overfit, therefore also have to tune to find the mid-point.| 
| `'min_samples_split'` |It controls the minimum size of a node before spliting. By tuning it can prevent the model from learning overly specific patterns,which reducing overfitting, however to generalised will lead to underfit, therfore it needs to be tuned.|
| `'min_samples_leaf'` |It restrict that leaf nodes contain more than a minimum number of samples, which improve model's ability to generalize, but needs to be tune as too high in this parameter will lead to underfitting as well.|
| `'max_features'` |Finds an optimal number that allows the trees to be diverse enough to capture various aspects of the data, while this will have risk in captureing noise, therefore needs to be tuned.|

Hyperparameters pre defined by me:

| Hyperparameters name |Reason of chose|number_choice|
|:-----------:|:-----------:|:-----------:|
|`'random_state'`|It ensures reproducibility which made the result more consisnent, ruther optimising results.|6|


In [ ]:
param_grid = {
    'RC__n_estimators': [100, 200, 300],
    'RC__max_depth': [10, 20, None],
    'RC__min_samples_split': [2, 5, 10],
    'RC__min_samples_leaf': [1, 2, 4],
    'RC__max_features': ['auto', 'sqrt'],
}

grid_search = GridSearchCV(
    pl, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

grid_search.fit(X_train[features], y_train)
print("Best parameters: ", grid_search.best_params_)
print("Best cross-validation accuracy: ", grid_search.best_score_)

In [ ]:
print(f'train score: {grid_search.score(X_train, y_train)}')
print(f'test score: {grid_search.score(X_test, y_test)}')
print(f'train rmse: {rmse(grid_search.predict(X_train), y_train)}')
print(f'test rmse: {rmse(grid_search.predict(X_test), y_test)}')

In [ ]:

ConfusionMatrixDisplay.from_estimator(grid_search, X_test, y_test)
plt.grid(False)
plt.savefig('graph/confusion_matrix.png')

In [ ]:
precision = precision_score(grid_search.predict(X_test), y_test)
recall = recall_score(grid_search.predict(X_test), y_test)

print(f"Precision: {precision}")
print(f"Recall: {recall}")

## Step 8: Fairness Analysis

run a **permutation test** to see if the difference in accuracy is significant.
- **Null Hypothesis**: The classifier's accuracy is the same for both MBC picked side and MBC not picked, and any differences are due to chance.
- **Alternative Hypothesis**: The classifier's accuracy is lower for MBC picked.
- **Test statistic**: Difference in accuracy (MBC picked side and MBC not picked).
- **Significance level**: 0.01.

In [ ]:
y_pred = grid_search.predict(X_test)
results = X_test
result['most_banned_champion_picked'] = all_pick_team['most_banned_champion_picked']
results['prediction'] = y_pred
results['actual'] = y_test
def compute_accuracy(x):
    return metrics.accuracy_score(x['actual'], x['prediction'])

accuracy_by_most_banned_champion_picked = results.groupby('most_banned_champion_picked').apply(
    compute_accuracy).rename('accuracy')
print(accuracy_by_most_banned_champion_picked)

In [ ]:
obs = accuracy_by_most_banned_champion_picked.diff().iloc[-1]
obs

In [ ]:
diff_in_acc = []
for _ in range(1000):
    s = (
        results[['most_banned_champion_picked', 'prediction', 'actual']]
        .assign(most_banned_champion_picked=np.random.permutation(results['most_banned_champion_picked']))
        .groupby('most_banned_champion_picked')
        .apply(compute_accuracy)
        .diff()
        .iloc[-1]
    )

    diff_in_acc.append(s)

In [ ]:
fig = pd.Series(diff_in_acc).plot(kind='hist', histnorm='probability', nbins=20,
                                  title='Empirical Distribution of Difference in Accuracy')
fig.add_vline(x=obs, line_color='red', line_width=2)
fig.add_annotation(text=f'<span style="color:red">Observed differences= {round(obsobserved_difference, 2)}</span>',
                   x=-.3 * obsobserved_difference, showarrow=False, y=0.17)
fig.update_xaxes(
    title_text="Differences in accuracy (MBC picked - MBC not picked)")

fig.write_html(
    'graph/diffference_acc.html')
fig

In [ ]:
(np.array(diff_in_acc) <= obs).mean()

P - value is 0.072

Therefore  **accept** the null hypothesis, that is The classifier's accuracy is the same for both MBC picked side and MBC not picked, and any differences are due to chance.Impliying there is not a statistically significant difference between the accuracy of MBC picked is  `True` and `False` from our model.